In [42]:
import kagglehub
import os
import pandas as pd

# 1. Iremos utilizar API DO  Kaggle para fazer os download dos dataset que sao os dados que irão alimentar nossa IA
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Caminho do dataset:", path)

# 2. Nesse techo vamos Pegar o arquivo CSV que ele acabou de baixar baixar via API, e garante que codigo localiza esse arquivo.
arquivos = os.listdir(path)
caminho_completo = os.path.join(path, arquivos[0])


#DATAFRAME: E transformar em dataframe(Dataframe e uma estrutura de dados bidimensional  semelhanre sql que organizamos a tabela linjas e colunas exatamente excel por exemplo.) a tabela e mostra as 5 primeiras linhas.

# 3. Nesse trecho vamos ler os arquivos de spam no csv atraves da biblioteca pandas,
df = pd.read_csv(caminho_completo, encoding='latin-1')
df.head()

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Caminho do dataset: /kaggle/input/sms-spam-collection-dataset


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
#1. Vai remover as colunas vazias  ou inuteis da nossa tabela. 
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

# 2. Aqui vamos Renomear as colunas 'v1' e 'v2'
df = df.rename(columns={'v1': 'label', 'v2': 'text'})

# 3.  Aqui vamos Criamos a coluna numérica que a matemática da IA precisa (0 = Normal(ham), 1 = Spam)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# 4. Verificamos se existe alguma linha com valor vazio (Nulo/NaN)
print("Quantidade de valores vazios por coluna:")
print(df.isnull().sum())

# 5.vamos  Mostrar a tabela ja concluirda processo ETL
print("\nTabela limpa e pronta para a IA:")
df.head()

Quantidade de valores vazios por coluna:
label        0
text         0
label_num    0
dtype: int64

Tabela limpa e pronta para a IA:


,label,text,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# 1). Separar texto e gabarito e dividir 80/20 (80% treino, 20% teste)
X = df['text']
y = df['label_num']
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# 2). O PIPELINE (A NOSSA LINHA DE MONTAGEM Semelhante linha montagem de montadora de carro. Aqui E O CÉREBRO IA!)
# O Pipeline é um fluxo automatizado. Em vez de limpar, transformar e prever separadamente,
# ele empacota tudo. O texto entra de um lado, passa pelos 3 passos abaixo, e sai classificado.
# Em resumo e um conjunto de regras e passos empacotado regra que se assemelha CI/CD.
pipeline = Pipeline([
    # PASSO 1: VECTORIZER (O Tradutor e Limpador)
    # CountVectorizer: Converte o texto para linguagem de máquina (números/vetores).
    # - stop_words='english': Trocamos a nossa função manual por este limpador nativo. Ele remove palavras inúteis ("the", "is") e aumenta a precisão.
    # - ngram_range=(1, 2): Ensina a IA a ler não só palavras isoladas, mas também duplas de palavras ("verify password", "exclusive offer").
    ('vectorizer', CountVectorizer(stop_words='english', ngram_range=(1, 2))),

    # PASSO 2: TF-IDF (O Balança de Pesos nas palavras.Exemplo: "winner", free, urgent) com base nos datasets ele aplica pesos nessa palavras.
    # TfidfTransformer: É usado para identificar a importância real das palavras.
    # Ele reduz o peso matemático de palavras muito comuns e aumenta a importância de palavras raras/suspeitas.
    # Por exemplo: palavras como "FREE", "winner" ou "urgent" ganham um peso enorme na identificação de spam.
    ('tfidf', TfidfTransformer()),


    # PASSO 3: CLASSIFIER (O Motor Matemático Agressivo)
    # LinearSVC: Substituímos o Naive Bayes (MultinomialNB) pelo Support Vector Classification.
    # Em vez de usar probabilidade simples, ele usa geometria para desenhar uma fronteira rígida (um muro)
    # entre o que é HAM e o que é SPAM. É um algoritmo "caçador de fraudes" muito mais agressivo,
    # que não deixa os golpes passarem batidos (reduzindo os Falsos Negativos). max_iter=5000 garante que ele tenha tempo de calcular essa fronteira.
    # analogia LinearSVC: ele funciona semelhante alemanha ocidental(spam) e oriental(ham) temos muro(hiperplano que linha reta(LinearSVC)) que separar as alemanha de um lado spam ou ham. e "margem" que zona segura que nao tem ninguem.E temos a Esteira de Decisão quando email chega ele passa na alfandega CountVectorizer + TF-IDF decidir com base nos datasets que nossa IA foi treinado e jogar lado do muro do spam ou ham. Assim gerando maior precisao do que naive bayes.
    ('classifier', LinearSVC(max_iter=5000)) # O novo algoritmo caçador de fraudes que implementamos na troca naive bayes.
])

# 3. Treina e testa tudo de uma vez
# Previsoes(predict) :"IA, tente adivinhar os emails de teste" porque atraves do email de teste que ela nunca viu antes aprender e jogar direto no spam.
pipeline.fit(X_treino, y_treino)
previsoes = pipeline.predict(X_teste)

print(f"A precisão do nosso NOVO MOTOR (LinearSVC) foi de: {accuracy_score(y_teste, previsoes) * 100:.2f}%")

A precisão do nosso NOVO MOTOR (LinearSVC) foi de: 97.58%


In [45]:
# Textos de teste em inglês que linguagem natural da IA
meus_emails = [
    "URGENT! You won a free iPhone 16 Pro Max. Click here now!",
    "Congratulations! You have been selected to receive a $1000 gift card.",
    "Txt 'WIN' to 80082 to claim your free holiday now!",
    "Your mobile number has won £5000 in the National Lottery. Call 087124 to claim.",
    "Hey bro, are we still going to the game tonight?"
]

# Passamos a lista de textos direto para o Pipeline
previsoes_novas = pipeline.predict(meus_emails)

print("--- TESTE DA NOSSA IA COM PIPELINE ---")
for email, previsao in zip(meus_emails, previsoes_novas):
    resultado = "SPAM (Lixo) 🚨" if previsao == 1 else "HAM (Normal) ✅"
    print(f"\nE-mail: '{email}'")
    print(f"Veredito da IA: {resultado}")

--- TESTE DA NOSSA IA COM PIPELINE ---

E-mail: 'URGENT! You won a free iPhone 16 Pro Max. Click here now!'
Veredito da IA: SPAM (Lixo) 🚨

E-mail: 'Congratulations! You have been selected to receive a $1000 gift card.'
Veredito da IA: SPAM (Lixo) 🚨

E-mail: 'Txt 'WIN' to 80082 to claim your free holiday now!'
Veredito da IA: SPAM (Lixo) 🚨

E-mail: 'Your mobile number has won £5000 in the National Lottery. Call 087124 to claim.'
Veredito da IA: SPAM (Lixo) 🚨

E-mail: 'Hey bro, are we still going to the game tonight?'
Veredito da IA: HAM (Normal) ✅


In [46]:
import joblib

# Exportamos apenas o Pipeline QUE IREMOS USAR NA "API" QUE IREMOS EXPORTAR NO PROJETO.
# ARQUIVO PKL e um arquivo binario gerado python que usado para salvar objetos no codigo. Por exemplo modelos de inteligencia aritificial) diretamento nosso disco rigido.
joblib.dump(pipeline, 'modelo_ia_spam.pk')

print("IA exportada com sucesso! O arquivo 'modelo_ia_spam.pkl' foi gerado.")

IA exportada com sucesso! O arquivo 'modelo_ia_spam.pkl' foi gerado.
